In [2]:
# Environment check (quiet)
import sys, os

print("Python:", sys.version.splitlines()[0])
print("Working directory:", os.getcwd())


Python: 3.12.7 (main, Oct  1 2024, 02:05:46) [Clang 16.0.0 (clang-1600.0.26.3)]
Working directory: /Users/tavishisharma/FlightDelay


In [4]:
import os, glob
import pandas as pd
import numpy as np

print("Working dir:", os.getcwd())
csv_candidates = glob.glob("*.csv")
print("CSV files found:", csv_candidates)

loaded_df = None
if csv_candidates:
    # Try to load the first CSV found and show a sample
    path = csv_candidates[0]
    try:
        df = pd.read_csv(path)
        print(f"\nLoaded first CSV: {path}  (shape: {df.shape})")
        display(df.head(5))
        loaded_df = df
    except Exception as e:
        print(f"\nError loading {path}: {e}")
        loaded_df = None

if loaded_df is None:
    # No usable CSV found — create a synthetic dataset, save it to dataset.csv, and show a sample
    print("\nNo usable CSV loaded. Creating a synthetic dataset and saving to 'dataset.csv'...")
    np.random.seed(42)
    n = 3000
    data = pd.DataFrame({
        "dep_hour": np.random.randint(0, 24, n),
        "distance": np.random.randint(200, 3000, n),
        "wind_speed": np.round(np.random.uniform(0, 40, n), 2),
        "visibility": np.round(np.random.uniform(1, 10, n), 2),
        # add a small categorical example column
        "carrier": np.random.choice(["AA","DL","UA","SW"], n)
    })
    # Probabilistic target with noise (not a trivial perfect rule)
    logits = (
        (data["dep_hour"] > 18).astype(int) * 0.6 +
        (data["wind_speed"] > 25).astype(int) * 0.5 +
        (data["visibility"] < 3).astype(int) * 0.7 +
        (data["distance"] > 1500).astype(int) * 0.15 +
        np.random.normal(0, 0.6, n)
    )
    probs = 1 / (1 + np.exp(-(logits - 0.8)))
    data["delayed"] = (probs > 0.5).astype(int)
    # save
    out_path = "dataset.csv"
    data.to_csv(out_path, index=False)
    print(f"Synthetic dataset saved to {out_path} (shape: {data.shape})")
    display(data.head(5))
    loaded_df = data

# Save a preview for later cells
loaded_df.head(0).to_csv("dataset_schema_preview.csv", index=False)


Working dir: /Users/tavishisharma/FlightDelay
CSV files found: []

No usable CSV loaded. Creating a synthetic dataset and saving to 'dataset.csv'...
Synthetic dataset saved to dataset.csv (shape: (3000, 6))


,dep_hour,distance,wind_speed,visibility,carrier,delayed
0,6,1371,5.43,3.30,AA,0
1,19,1695,35.75,8.73,AA,1
2,14,1143,2.99,5.92,DL,0
3,10,2446,36.31,6.13,AA,0
4,7,1649,39.86,4.09,AA,0


In [3]:
# Cell 2 — Simple synthetic delay prediction model

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# -----------------------------
# 1. Create synthetic dataset
# -----------------------------
np.random.seed(42)

n_samples = 2000

data = pd.DataFrame({
    "dep_hour": np.random.randint(0, 24, n_samples),
    "distance": np.random.randint(200, 3000, n_samples),
    "wind_speed": np.random.uniform(0, 40, n_samples),
    "visibility": np.random.uniform(1, 10, n_samples)
})

# Simple rule-based target (synthetic logic)
data["delayed"] = (
    (data["dep_hour"] > 18) |
    (data["wind_speed"] > 25) |
    (data["visibility"] < 3)
).astype(int)

# -----------------------------
# 2. Train model
# -----------------------------
X = data.drop(columns=["delayed"])
y = data["delayed"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)

print("Model trained.")
print("Test Accuracy:", round(accuracy, 3))

Model trained.
Test Accuracy: 1.0
